# SEE STATS FOR CLEAR SKY DAYS AND TIMESTAMPS FOR CLEAR SKY DAY
# THIS IS FOR structured_data_5m.parquet

In [ ]:
import sys
from pathlib import Path

import polars as pl


def find_curtailment_scripts_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        direct = candidate / "SAPN2022_Analysis" / "curtailment" / "scripts"
        if (direct / "path_config.py").exists():
            return direct
        if (candidate / "path_config.py").exists() and candidate.name == "scripts":
            return candidate
    raise RuntimeError(
        "Could not locate SAPN2022_Analysis/curtailment/scripts from the current working directory."
    )


SCRIPTS_DIR = find_curtailment_scripts_dir(Path.cwd().resolve())
PROJECT_ROOT = SCRIPTS_DIR.parent

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

# Local SAPN/BOM paths live in `local_paths.py` so this tracked notebook does
# not commit machine-specific local path values. Start from
# `local_paths.example.py` if you need to create the local file.
from path_config import require_local_path

BOM_ROOT = require_local_path(
    "BOM_ROOT",
    "root folder containing the BOM daily parquet files.",
)

# Read the 5-minute structured dataset used in this notebook.
structured_5m_path = (
    PROJECT_ROOT / "outputs" / "all_structured_data_5m" / "structured_data_5m.parquet"
)
structured_5m = pl.scan_parquet(structured_5m_path)

# Summarise clear-sky availability for each site.
site_summary = (
    structured_5m.group_by("site_id")
    .agg(
        [
            pl.first("n_lat").alias("n_lat"),
            pl.first("n_long").alias("n_long"),
            pl.col("cs_day").is_not_null().any().alias("has_clear_sky_day"),
            pl.col("P_kw_norm_cs")
            .is_not_null()
            .any()
            .alias("has_usable_clear_sky_power"),
        ]
    )
    .sort("site_id")
    .collect()
)

# Keep the sites that never received a clear-sky day.
missing_clear_sky_sites = site_summary.filter(~pl.col("has_clear_sky_day"))

# Print the site counts requested in the notebook comments.
summary_counts = site_summary.select(
    [
        pl.len().alias("total_sites"),
        pl.col("has_clear_sky_day").sum().alias("sites_with_clear_sky_day"),
        pl.col("has_usable_clear_sky_power")
        .sum()
        .alias("sites_with_usable_clear_sky_power"),
        (~pl.col("has_clear_sky_day")).sum().alias("sites_with_no_clear_sky_day"),
        (pl.col("has_clear_sky_day") & ~pl.col("has_usable_clear_sky_power"))
        .sum()
        .alias("sites_with_clear_sky_day_but_no_usable_power"),
    ]
)

print(structured_5m_path)
print(summary_counts)

In [ ]:
import build_structured_high_resolution as bsl

# Use the BOM files for the sites that never received a clear-sky day.
missing_site_ids = missing_clear_sky_sites["site_id"].to_list()

train_days_for_missing_sites = (
    structured_5m.filter(
        (pl.col("dataset_role") == "train") & pl.col("site_id").is_in(missing_site_ids)
    )
    .select(["site_id", "n_lat", "n_long", "actual_day"])
    .unique()
    .collect()
)

best_bom_day = pl.DataFrame(
    schema={
        "site_id": pl.Int64,
        "n_lat": pl.Float64,
        "n_long": pl.Float64,
        "best_available_day": pl.Date,
        "cloud_sum": pl.Float64,
        "max_GHI": pl.Float64,
    }
)

if not train_days_for_missing_sites.is_empty():
    train_start_day = train_days_for_missing_sites["actual_day"].min()
    train_end_day = train_days_for_missing_sites["actual_day"].max()
    bom_mapping = missing_clear_sky_sites.select(
        ["site_id", "n_lat", "n_long"]
    ).unique()
    bom_files = bsl.bom_daily_parquets(BOM_ROOT, train_start_day, train_end_day)

    # Summarise BOM cloud cover and maximum GHI by day.
    bom_daily = (
        bsl.prepare_bom10min(bom_files, bom_mapping)
        .with_columns(bsl.adelaide_datetime_expr("time").dt.date().alias("actual_day"))
        .group_by(["latitude", "longitude", "actual_day"])
        .agg(
            [
                pl.col("cloud_type").sum().alias("cloud_sum"),
                pl.col("GHI").max().alias("max_GHI"),
            ]
        )
        .collect()
    )

    # Keep the best available BOM day for each site.
    best_bom_day = (
        train_days_for_missing_sites.join(
            bom_daily,
            left_on=["n_lat", "n_long", "actual_day"],
            right_on=["latitude", "longitude", "actual_day"],
            how="left",
        )
        .sort(
            ["site_id", "cloud_sum", "max_GHI", "actual_day"],
            descending=[False, False, True, False],
            nulls_last=True,
        )
        .group_by("site_id")
        .agg(
            [
                pl.first("n_lat").alias("n_lat"),
                pl.first("n_long").alias("n_long"),
                pl.first("actual_day").alias("best_available_day"),
                pl.first("cloud_sum").alias("cloud_sum"),
                pl.first("max_GHI").alias("max_GHI"),
            ]
        )
    )

# Print the sites with no clear-sky day and the best BOM conditions they had.
no_clear_sky_details = (
    missing_clear_sky_sites.join(
        best_bom_day, on=["site_id", "n_lat", "n_long"], how="left"
    )
    .with_columns(
        [
            pl.col("best_available_day").is_not_null().alias("has_train_rows"),
            (pl.col("cloud_sum") <= 60).alias("passes_cloud_filter"),
            (pl.col("max_GHI") > 200).alias("passes_ghi_filter"),
        ]
    )
    .select(
        [
            "site_id",
            "n_lat",
            "n_long",
            "has_train_rows",
            "best_available_day",
            "cloud_sum",
            "max_GHI",
            "passes_cloud_filter",
            "passes_ghi_filter",
        ]
    )
    .sort("site_id")
)

print("Clear-sky candidate rule: cloud_sum <= 60 and max_GHI > 200")
print(no_clear_sky_details)

In [ ]:
from datetime import time

# There are 144 five-minute bins between 6am and 6pm.
daylight_start = time(6, 0)
daylight_end = time(18, 0)
total_daylight_5m_bins = int((12 * 60) / 5)

# Count how many daylight bins each site-day actually contains.
site_days = structured_5m.select(["dataset_role", "site_id", "actual_day"]).unique()

daylight_counts = (
    structured_5m.filter(
        (pl.col("actual_tod") >= pl.lit(daylight_start))
        & (pl.col("actual_tod") < pl.lit(daylight_end))
    )
    .group_by(["dataset_role", "site_id", "actual_day"])
    .agg(pl.col("actual_tod").n_unique().alias("bins_with_data"))
)

# Calculate how many bins are missing for each site-day.
daylight_bin_counts = (
    site_days.join(
        daylight_counts, on=["dataset_role", "site_id", "actual_day"], how="left"
    )
    .with_columns(pl.col("bins_with_data").fill_null(0))
    .with_columns(
        (pl.lit(total_daylight_5m_bins) - pl.col("bins_with_data")).alias(
            "missing_bins"
        )
    )
    .sort(["missing_bins", "site_id", "actual_day"], descending=[True, False, False])
    .collect()
)

# Show the site-days where daylight coverage is incomplete.
incomplete_daylight_days = daylight_bin_counts.filter(
    pl.col("bins_with_data") < total_daylight_5m_bins
)

print(f"Total 5-minute bins between 6am and 6pm: {total_daylight_5m_bins}")
print(f"Site-days with missing daylight bins: {incomplete_daylight_days.height}")
print(incomplete_daylight_days)

In [ ]:
from datetime import datetime, timedelta

import matplotlib.pyplot as plt

# Run the previous cell first so the daylight summary tables exist.
if "daylight_bin_counts" not in globals():
    raise RuntimeError(
        "Run the previous cell first so daylight_bin_counts is available."
    )

# Only use site-days that have some daylight data in the structured file.
site_days_with_daylight_data = daylight_bin_counts.filter(
    pl.col("bins_with_data") > 0
).select(["dataset_role", "site_id", "actual_day"])

# Build the full set of expected 5-minute times from 6am to 6pm.
expected_times = []
current_time = datetime.combine(datetime.today().date(), daylight_start)
end_time = datetime.combine(datetime.today().date(), daylight_end)
while current_time < end_time:
    expected_times.append(current_time.time())
    current_time += timedelta(minutes=5)

expected_daylight_bins = pl.DataFrame({"actual_tod": expected_times})

# Find which 5-minute times are missing for each site-day.
observed_daylight_bins = (
    structured_5m.filter(
        (pl.col("actual_tod") >= pl.lit(daylight_start))
        & (pl.col("actual_tod") < pl.lit(daylight_end))
    )
    .select(["dataset_role", "site_id", "actual_day", "actual_tod"])
    .unique()
    .collect()
)

missing_daylight_times = (
    site_days_with_daylight_data.lazy()
    .join(expected_daylight_bins.lazy(), how="cross")
    .join(
        observed_daylight_bins.lazy(),
        on=["dataset_role", "site_id", "actual_day", "actual_tod"],
        how="anti",
    )
    .collect()
)

# Count how often each clock time is missing.
missing_time_distribution = (
    missing_daylight_times.group_by("actual_tod")
    .len()
    .sort("actual_tod")
    .with_columns(pl.col("actual_tod").dt.strftime("%H:%M").alias("time_label"))
)

if missing_time_distribution.is_empty():
    print("No missing daylight bins found.")
else:
    print(missing_time_distribution)

    plt.figure(figsize=(12, 4))
    plt.bar(
        missing_time_distribution["time_label"].to_list(),
        missing_time_distribution["len"].to_list(),
        color="steelblue",
        width=0.9,
    )
    plt.xticks(rotation=90)
    plt.xlabel("Time of day")
    plt.ylabel("Number of site-days missing this 5-minute bin")
    plt.title("Missing daylight 5-minute bins by time of day")
    plt.tight_layout()
    plt.show()